# TimesFM 2.5 — Load


In [ ]:
# !pip install -r requirements/timesfm.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

import timesfm
from preprocessing import load_data, describe_data, SITES, CONTEXT_LENGTH, PREDICTION_LENGTH
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=BASE_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data=load_data(ROOT/"data")
print(describe_data(data))


## GEFCom2014-L


In [ ]:
# GEFCom2014-L (cleaned input; no extra cleaning)
from gefcom import load_gefcom_load, load_initial_and_tasks, evaluate_gefcom_load, LOAD_WEATHER
gef_load=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")
gef_context, gef_tasks=load_initial_and_tasks(gef_load)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
gef_model=timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
gef_model.compile(timesfm.ForecastConfig(max_context=CONTEXT_LENGTH,max_horizon=744,normalize_inputs=True,use_continuous_quantile_head=True,force_flip_invariance=True,infer_is_positive=True,fix_quantile_crossing=True))

# GEFCom2014-L temperatures are past-only; this TimesFM interface therefore uses target history only here.
rolling=gef_context.copy(); rows=[]
for task_id,future in tqdm(gef_tasks):
    hist=rolling.sort_values("timestamp").target.to_numpy()[-CONTEXT_LENGTH:]; H=len(future)
    point,qhat=gef_model.forecast(horizon=H,inputs=[hist]); point=np.asarray(point)[0][-H:]; qhat=np.asarray(qhat)[0][-H:]
    for h,(_,r) in enumerate(future.iterrows()):
        row={"item_id":r.item_id,"timestamp":r.timestamp,"task_id":task_id,"prediction":point[h]}
        for qi,q in enumerate(QUANTILES): row[str(q)]=qhat[h,qi+1] if qhat.shape[1]>len(QUANTILES) else qhat[h,qi]
        rows.append(row)
    rolling=pd.concat([rolling,future],ignore_index=True)
gef_result=pd.DataFrame(rows).merge(gef_load[["item_id","timestamp","target","task_id"]],on=["item_id","timestamp","task_id"],how="inner")
gef_result.to_csv(GEF_OUTPUT/"timesfm_gefcom_load.csv",index=False)
gef_summary=evaluate_gefcom_load(gef_result,QUANTILES); display(gef_summary[["task_id","raw_aql","normalized_aql"]])
gef_summary.to_csv(GEF_OUTPUT/"gefcom_load_summary.csv",index=False)


## PG&E competition


In [ ]:
VARIANT="with_covariates"
FUTURE_COLS=data.covariates
model=timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(max_context=CONTEXT_LENGTH,max_horizon=PREDICTION_LENGTH,normalize_inputs=True,use_continuous_quantile_head=True,force_flip_invariance=True,infer_is_positive=True,fix_quantile_crossing=True))


In [ ]:
rolling={s:data.context_by_site[s].copy() for s in data.context_by_site}
rows=[]
for i in tqdm(range(len(next(iter(data.daily_by_site.values()))))):
    day={s:data.daily_by_site[s][i][1].sort_values("timestamp").iloc[:PREDICTION_LENGTH].copy() for s in data.daily_by_site}
    for site,future in day.items():
        context=rolling[site].sort_values("timestamp"); hist=context.target.to_numpy()[-CONTEXT_LENGTH:]
        if VARIANT=="with_covariates":
            dynamic={c:[np.concatenate([context[c].to_numpy()[-CONTEXT_LENGTH:],future[c].to_numpy()])] for c in FUTURE_COLS}
            point,qhat=model.forecast_with_covariates(inputs=[hist],dynamic_numerical_covariates=dynamic,dynamic_categorical_covariates={},static_numerical_covariates={},static_categorical_covariates={},xreg_mode="timesfm + xreg",ridge=0.0,normalize_xreg_target_per_input=True)
        else:
            point,qhat=model.forecast(horizon=PREDICTION_LENGTH,inputs=[hist])
        point=np.asarray(point)[0][-PREDICTION_LENGTH:]; qhat=np.asarray(qhat)[0][-PREDICTION_LENGTH:]
        for h,ts in enumerate(future.timestamp.iloc[:PREDICTION_LENGTH]):
            row={"item_id":future.item_id.iloc[0],"timestamp":ts,"prediction":float(point[h])}
            for qi,q in enumerate(MODEL_QUANTILES): row[str(q)]=float(qhat[h,qi+1])
            rows.append(row)
    for site in day: rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True).sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
result=pd.DataFrame(rows).merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"timesfm_{VARIANT}.csv",index=False)
print_metrics(result,QUANTILES,label=f"timesfm_{VARIANT}")
